In [1]:
using Clapeyron, Metaheuristics, Printf

In [58]:
#Hfus dan Tm masukin ke csv yh jgn lupa

like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
alanine,89.09,6.2636,2.4635,264.4748,2,2
water,18.02,1.2047,2.801457,353.94,1,1
"""

unlike_parameter = """Clapeyron Database File
PCSAFT Unlike Parameters [csvtype = unlike,grouptype = PCSAFT]
species1,species2,k
water,alanine,-0.0612
"""

#jgn lupa combing rules yh
assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
water,H,water,e,2425.67,0.045
alanine,H,alanine,e,3176.6,0.0819
water,H,alanine,e,2801.135,0.060624739
water,e,alanine,H,2801.135,0.060624739
"""
components = ["water", "alanine"]
model = CompositeModel(components;
                       fluid = PCSAFT,
                       solid = SolidHfus,
                       fluid_userlocations = [like_parameter, unlike_parameter, assoc_parameter])

println(model.fluid.params.epsilon.values)
println(model.fluid.params.sigma.values)
println("======================")
println(model.fluid.params.epsilon_assoc.values)
println(model.fluid.params.bondvol.values)
println("kij = ", (1  - ((model.fluid.params.epsilon.values[2])/(sqrt(model.fluid.params.epsilon.values[1] * model.fluid.params.epsilon.values[4])))))

[353.94 324.6790101621568; 324.6790101621568 264.4748]
[2.8014570000000003e-10 2.6324785000000003e-10; 2.6324785000000003e-10 2.4635e-10]
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2425.67, 2801.135, 2801.135, 3176.6]
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[0.045, 0.060624739, 0.060624739, 0.0819]
kij = -0.06119999999999992


In [59]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("gamma_alanine.csv")
fix_line_endings("rhol_alanine.csv")

Fixed: gamma_alanine.csv
Fixed: rhol_alanine.csv


In [60]:
#SOLUTION DENSITY

Mw_water = model.fluid.params.Mw[1] / 1000
Mw_aa = model.fluid.params.Mw[2] / 1000

println("MW Solute  : ", Mw_aa, " kg/m3")
println("MW solvent : ", Mw_water, " kg/m3")

function molality_to_z(m::Float64, Mw_solute::Float64)
    # m mol solute per kg water
    n_solute = m
    n_water  = 1.0 / Mw_water   # mol water per kg water = 55.51
    x_water  = n_water / (n_water + n_solute)
    x_solute = n_solute / (n_water + n_solute)
    return [x_water, x_solute]   # [water, amino acid]
end

function solution_density(model::EoSModel, m::Float64)
    z   = molality_to_z(m, Mw_aa)
    T   = 298.15
    p   = 101325.0
    V   = volume(model, p, T, z; phase=:liquid)        # m³/mol mixture
    Mw_mix = z[1]*Mw_water + z[2]*Mw_aa               # kg/mol
    return Mw_mix / V                                   # kg/m³
end

MW Solute  : 0.08909 kg/m3
MW solvent : 0.01802 kg/m3


solution_density (generic function with 1 method)

In [61]:
#ACTIVITY COEFFICIENT

function chem_activity_coeff(model::EoSModel, m::Float64)
    z     = molality_to_z(m, model.fluid.params.Mw[2])
    z_inf = molality_to_z(1e-10, model.fluid.params.Mw[2])   # very dilute reference
    p     = 101325.0
    T     = 298.15
    RT    = Rgas(model) * T

    # get chemical potentials directly
    chem_pot     = chemical_potential(model.fluid, p, T, z;     phase=:liquid)
    chem_pot_inf = chemical_potential(model.fluid, p, T, z_inf; phase=:liquid)

    # γ dari persamaan chemical potential
    gamma_star_x = exp((chem_pot[2] - chem_pot_inf[2]) / RT) * (z_inf[2] / z[2])

    # convert to molality basis (Kuramochi eq 5)
    #println("exp((μ[2] - μ_inf[2]) / RT) :", exp((μ[2] - μ_inf[2]) / RT))
    #println("m :",m)
    #println("z_inf :",z_inf)
    #println("z[2]: ",z[2])
    #println("z_inf[2] / z[2] :",z_inf[2] / z[2])
    return gamma_star_x / (1.0 + Mw_water * m)
end

chem_activity_coeff (generic function with 1 method)

In [69]:
toestimate = [
    Dict(
        :param => :epsilon,
        :indices => (1,2), #kij
        :lower => 200.,
        :upper => 400.,
        :guess => 300.
    ),
    #Dict(
     #   :param => :sigma,
      #  :indices => (2,2),
       # :factor => 1e-10,
        #:lower => 2.0,
        #:upper => 3.0,
        #:guess => 2.5
    #),
    Dict(
        :param   => :epsilon_assoc,
        :indices => 4,
        #:cross_assoc => true,
        :lower   => 3000.0,
        :upper   => 4000.0,
        :guess   => 3100.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => 4,
        #:cross_assoc => true,
        :lower   => 0.07,
        :upper   => 0.1,
        :guess   => 0.089
    ),
]

3-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 400.0, :param => :epsilon, :indices => (1, 2), :guess => 300.0, :lower => 200.0)
 Dict(:upper => 4000.0, :param => :epsilon_assoc, :indices => 4, :guess => 3100.0, :lower => 3000.0)
 Dict(:upper => 0.1, :param => :bondvol, :indices => 4, :guess => 0.089, :lower => 0.07)

In [70]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_alanine.csv",
        "gamma_alanine.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))
println(x0)

Initial objective value: 0.0426162290957047
[300.0, 3100.0, 0.089]


In [71]:
method = ECA(; options = Options(iterations = 10000, seed = 42))
 
params_opt, model_opt = optimize(objective, estimator, method)

([330.91758869566297, 3384.6058094636505, 0.099429479074925], CompositeModel{PCSAFT{BasicIdeal, Float64}, SolidHfus}("water", "alanine"))

In [72]:
println("\n=== Optimized PC-SAFT parameters for glycine ===")
println("  segment (m)  : ", model_opt.fluid.params.segment[2])
println("  sigma   (m)  : ", model_opt.fluid.params.sigma[2,2])
println("  epsilon (K1)  : ", model_opt.fluid.params.epsilon.values)
println("  epsilon_assoc  : ", model_opt.fluid.params.epsilon_assoc)
println("  bondvol  : ", model_opt.fluid.params.bondvol)
println("==================")
println(model_opt.fluid.params.epsilon.values)
println(model_opt.fluid.params.sigma.values)
println("kij = ", (1  - ((model_opt.fluid.params.epsilon.values[2])/(sqrt(model_opt.fluid.params.epsilon.values[1] * model_opt.fluid.params.epsilon.values[4])))))
println("======================")
println(model_opt.fluid.params.epsilon_assoc.values)
println(model_opt.fluid.params.bondvol.values)



=== Optimized PC-SAFT parameters for glycine ===
  segment (m)  : 6.2636
  sigma   (m)  : 2.4635e-10
  epsilon (K1)  : [353.94 330.91758869566297; 330.91758869566297 264.4748]
  epsilon_assoc  : AssocParam{Float64}("epsilon_assoc")[2425.67, 2801.135, 2801.135, 3384.6058094636505]
  bondvol  : AssocParam{Float64}("bondvol")[0.045, 0.060624739, 0.060624739, 0.099429479074925]
[353.94 330.91758869566297; 330.91758869566297 264.4748]
[2.8014570000000003e-10 2.6324785000000003e-10; 2.6324785000000003e-10 2.4635e-10]
kij = -0.08159053752335343
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2425.67, 2801.135, 2801.135, 3384.6058094636505]
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[0.045, 0.060624739, 0.060624739, 0.099429479074925]


In [73]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [74]:
aard_p   = calculate_AAD(model_opt, "rhol_alanine.csv", solution_density)


=== AAD: rhol_alanine.csv ===
Clapeyron Estimator  exp           calc          ARD%    
0.0316      997.957000    993.319867    0.4647  
0.0583      998.714000    993.978361    0.4742  
0.1044      999.965000    995.112360    0.4853  
0.1508      1001.290000   996.248354    0.5035  
0.2033      1002.753000   997.527212    0.5211  
0.2401      1003.715000   998.419550    0.5276  
0.2933      1005.212000   999.703649    0.5480  
0.3718      1007.344000   1001.585736   0.5716  
0.4047      1008.224000   1002.370066   0.5806  
AARD = 0.5196%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


0.5196208128267853

In [75]:
aard_p   = calculate_AAD(model_opt, "gamma_alanine.csv", chem_activity_coeff)


=== AAD: gamma_alanine.csv ===
Clapeyron Estimator  exp           calc          ARD%    
0.1013      0.990100      0.988654      0.1460  
0.1505      0.985300      0.983600      0.1725  
0.2007      0.980500      0.978734      0.1801  
0.2509      0.975800      0.974148      0.1692  
0.3502      0.966600      0.965863      0.0762  
0.4019      0.961900      0.961941      0.0042  
0.4505      0.957600      0.958485      0.0924  
0.5015      0.953000      0.955090      0.2193  
AARD = 0.1325%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


0.13250022122313238